In [ ]:
%pip install sagemaker==2.254.1 --upgrade --quiet --no-warn-conflicts

In [ ]:
import sys
import os
import time
import json
import boto3
import sagemaker

role = sagemaker.get_execution_role()  # execution role for the endpoint
sess = sagemaker.session.Session()  # sagemaker session for interacting with different AWS APIs
bucket = sess.default_bucket()  # bucket to house artifacts
region = sess._region_name  # region name of the current SageMaker Studio environment
account_id = sess.account_id()

sm_client = boto3.client("sagemaker")  # client to intreract with SageMaker
smr_client = boto3.client("sagemaker-runtime")  # client to intreract with SageMaker Endpoints
s3 = boto3.client("s3")

print(f"sagemaker role arn: {role}")
print(f"sagemaker bucket: {bucket}")
print(f"sagemaker session region: {region}")
print(f"boto3 version: {boto3.__version__}")
print(f"sagemaker version: {sagemaker.__version__}")

In [ ]:
model_id = "Wan-AI/Wan2.2-TI2V-5B-Diffusers"
s3_key = f"model/{model_id}"
s3_code_key = f"{s3_key}/code"

In [ ]:
from huggingface_hub import snapshot_download
from pathlib import Path

local_model_path = Path("./data_5b")
local_model_path.mkdir(exist_ok=True)

snapshot_download(repo_id=model_id, local_dir=local_model_path)

###### enumerate local files recursively
for root, dirs, files in os.walk(local_model_path):
    for filename in files:
        local_path = os.path.join(root, filename)

        relative_path = os.path.relpath(local_path, local_model_path)
        s3_path = os.path.join(s3_key, relative_path)

        print("Uploading %s..." % s3_path)
        s3.upload_file(local_path, bucket, s3_path)

### Files for Amazon Sagemaker inference ###
# SageMaker Endpoint — Required Files

SageMaker runs your model in a generic Docker container. It needs two files to know how to load and serve your model.

---

## `inference.py` — Entry point for requests

Defines four hooks SageMaker calls automatically:

```python
def model_fn(model_dir):        # Called once at startup — load model
    return torch.load(f"{model_dir}/model.pt")

def input_fn(body, content_type):   # Parse incoming request
    return json.loads(body)

def predict_fn(data, model):        # Run inference
    return model(data)

def output_fn(prediction, accept):  # Format response
    return json.dumps(prediction)
```

---

## `requirements.txt` — Dependencies

The base image includes common packages (torch, numpy) but not everything. SageMaker runs `pip install -r requirements.txt` before calling `model_fn`.

```
transformers==4.40.0
flash-attn==2.5.8
sentencepiece==0.2.0
accelerate==0.29.0
```

---

## Startup Sequence

```
Pull base image → pip install requirements.txt → model_fn() → endpoint live
```


def input_fn(body, content_type):   # Parse incoming request
    return json.loads(body)

def predict_fn(data, model):        # Run inference
    return model(data)

def output_fn(prediction, accept):  # Format response
    return json.dumps(prediction)
```

---

## `requirements.txt` — Dependencies

The base image includes common packages (torch, numpy) but not everything. SageMaker runs `pip install -r requirements.txt` before calling `model_fn`.

```
transformers==4.40.0
flash-attn==2.5.8
sentencepiece==0.2.0
accelerate==0.29.0
```

---

## Startup Sequence

```
Pull base image → pip install requirements.txt → model_fn() → endpoint live
```

In [ ]:
requirements = """torch>=2.4.0
torchvision>=0.19.0
torchaudio
opencv-python>=4.9.0.80
diffusers==0.36.0
peft>=0.17.0
transformers>=4.49.0,<=4.51.3
tokenizers>=0.20.3
accelerate>=1.1.1
tqdm
imageio[ffmpeg]
easydict
ftfy
dashscope
imageio-ffmpeg
numpy>=1.23.5,<2
"""
file_name = "requirements.txt"
with open(file_name, 'w') as f:
    f.write(requirements)
    
print(f"{s3_code_key}/{file_name}")

key = f"{s3_code_key}/{file_name}"
s3.upload_file(file_name, bucket, key)

In [ ]:
inference = """import os
os.environ["TRANSFORMERS_NO_FLASH_ATTN"] = "1"
import time
import boto3
import torch
import numpy as np
from io import BytesIO
from PIL import Image
from botocore.exceptions import ClientError
from diffusers import AutoencoderKLWan, WanImageToVideoPipeline
from diffusers.utils import export_to_video


def upload_file(file_name, bucket, object_name=None):
    if object_name is None:
        object_name = os.path.basename(file_name)

    s3 = boto3.client('s3')
    try:
        s3.upload_file(file_name, bucket, object_name)
    except ClientError as e:
        print(e)
        return False
    return True


def load_image_from_payload(data):
    
    #Accepts an image from one of three sources in the request payload:
    #  - image_s3_uri  : "s3://bucket/key"  (recommended for large images)
    #  - image_url     : any public HTTPS URL
    #  - image_bytes   : raw bytes already loaded by the serving stack
    #Returns a PIL.Image in RGB.
    
    s3 = boto3.client('s3')

    if "image_s3_uri" in data:
        uri = data.pop("image_s3_uri")                  # e.g. "s3://my-bucket/input.jpg"
        bucket, key = uri.replace("s3://", "").split("/", 1)
        buf = BytesIO()
        s3.download_fileobj(bucket, key, buf)
        buf.seek(0)
        return Image.open(buf).convert("RGB")

    if "image_url" in data:
        from diffusers.utils import load_image as diffusers_load_image
        return diffusers_load_image(data.pop("image_url"))  # already RGB

    if "image_bytes" in data:
        buf = BytesIO(data.pop("image_bytes"))
        return Image.open(buf).convert("RGB")

    raise ValueError("Payload must contain one of: image_s3_uri, image_url, image_bytes")


def model_fn(model_dir):
    
    print("Loading Wan2.2 TI2V model...")

    # VAE in fp32 for numerical stability (same as reference code)
    vae = AutoencoderKLWan.from_pretrained(
        model_dir, subfolder="vae", torch_dtype=torch.float32
    )
    
    pipe = WanImageToVideoPipeline.from_pretrained(
        model_dir,
        vae=vae,
        torch_dtype=torch.bfloat16)
        
    pipe.to("cuda")
    print("Model loaded successfully.")
    return pipe


def predict_fn(data, pipe):
    
    print("Inference started")

    # ── Output / storage params ───────────────────────────────────────────────
    bucket    = data.pop("bucket")
    file_name = data.pop("file_name", "i2v_output.mp4")

    # ── Load input image (required) ───────────────────────────────────────────
    image = load_image_from_payload(data)

    # ── Generation params (all optional with sensible defaults) ───────────────
    prompt = data.pop(
        "prompt",
        "A cinematic, smoothly animated scene"
    )
    negative_prompt = data.pop(
        "negative_prompt",
        "色调艳丽，过曝，静态，细节模糊不清，字幕，风格，作品，画作，画面，静止，整体发灰，"
        "最差质量，低质量，JPEG压缩残留，丑陋的，残缺的，多余的手指，画得不好的手部，"
        "画得不好的脸部，畸形的，毁容的，形态畸形的肢体，手指融合，静止不动的画面，"
        "杂乱的背景，三条腿，背景人很多，倒着走"
    )
    num_frames      = int(data.pop("num_frames",      81))
    guidance_scale  = float(data.pop("guidance_scale", 7.5))
    inference_steps = int(data.pop("num_inference_steps", 50))
    fps             = int(data.pop("fps",              16))
    seed            = data.pop("seed", 0)

    # ── Compute height/width from image aspect ratio ──────────────────────────
    # Allow caller to override; otherwise auto-derive from the input image.
    if "height" in data and "width" in data:
        height = int(data.pop("height"))
        width  = int(data.pop("width"))
        image  = image.resize((width, height))
    else:
        data.pop("height", None)
        data.pop("width",  None)
        max_area     = int(data.pop("max_area", 480 * 832))
        aspect_ratio = image.height / image.width
        mod_value    = (
            pipe.vae_scale_factor_spatial
            * pipe.transformer.config.patch_size[1]
        )
        height = round(np.sqrt(max_area * aspect_ratio)) // mod_value * mod_value
        width  = round(np.sqrt(max_area / aspect_ratio)) // mod_value * mod_value
        image  = image.resize((width, height))

    print(f"Generating {num_frames} frames at {width}x{height} | prompt: {prompt[:80]}…")

    # ── Run pipeline ──────────────────────────────────────────────────────────


    start_time = time.perf_counter()

    output = pipe(
        image=image,
        prompt=prompt,
        negative_prompt=negative_prompt,
        height=height,
        width=width,
        num_frames=num_frames,
        guidance_scale=guidance_scale,
        num_inference_steps=inference_steps
    ).frames[0]

    elapsed = time.perf_counter() - start_time
    print(f"Inference time: {elapsed:.2f}s")

    # ── Save & upload ─────────────────────────────────────────────────────────
    file_path = f"/tmp/{os.path.basename(file_name)}"
    export_to_video(output, file_path, fps=fps)
    upload_file(file_path, bucket, file_name)

    try:
        os.remove(file_path)
    except FileNotFoundError:
        print(f"Warning: temp file '{file_path}' already gone.")
    except Exception as e:
        print(f"Cleanup error: {e}")
        
    print(f"s3://{bucket}/{file_name}")
    
    return {"generated_video": f"s3://{bucket}/{file_name}"}
"""
file_name = "inference.py"
with open(file_name, 'w') as f:
    f.write(inference)

key = f"{s3_code_key}/{file_name}"
s3.upload_file(file_name, bucket, key)

In [ ]:
inference_image = f"763104351884.dkr.ecr.{region}.amazonaws.com/huggingface-pytorch-inference:2.6.0-transformers4.49.0-gpu-py312-cu124-ubuntu22.04"

# 5. Use the image — swap into your existing deployment script
#inference_image = ecr_image_uri
print(f"Ready to deploy with: {inference_image}")

instance = {"type": "ml.g6e.2xlarge", "num_gpu": 1}
model_name = f"model-{time.strftime('%y%m%d-%H%M%S')}"
endpoint_name = f"{model_name}"
endpoint_config_name = model_name
timeout = 600
variant_name = "main"

model_data_source = {
    'S3DataSource': {
        'S3Uri': f"s3://{bucket}/{s3_key}/",
        'S3DataType': 'S3Prefix',
        'CompressionType': 'None',
    }
}

In [ ]:
model_response = sm_client.create_model(
    ModelName=model_name,
    ExecutionRoleArn=role,
    PrimaryContainer={
        "Image": inference_image,
        "ModelDataSource": model_data_source
    },
)

# SageMaker Endpoints

## Required Files

| File | Purpose |
|---|---|
| `inference.py` | Tells SageMaker how to load your model and handle requests |
| `requirements.txt` | Lists extra packages to install before startup |

---

## Synchronous Endpoint

The default. Client sends a request and **waits for the response**.

- Timeout: **60 seconds**
- Good for: fast, real-time predictions

---

## Asynchronous Endpoint

Client sends a request, SageMaker **queues it and writes the result to S3** when done.

- Timeout: **up to 15 minutes**
- Good for: slow models, large payloads, infrequent workloads
- Supports **scale-to-zero** (no cost when idle)

---

## When to Use Which

```
Fast inference (<60s)   →  Synchronous
Slow inference (>60s)   →  Asynchronous
Infrequent traffic      →  Asynchronous (scale to zero saves cost)
```


## SNS for Async Endpoints

Instead of polling S3 for the result, SNS **pushes a notification** to your app when inference is done (or fails).

| Without SNS | With SNS |
|---|---|
| You poll S3 repeatedly | SageMaker notifies you automatically |
| Wastes compute | Event-driven, efficient |

SageMaker publishes to two topics you provide at endpoint creation:
- **Success topic** — triggered when output is written to S3
- **Error topic** — triggered on failure

**Reference:** [AWS Docs — Create an SNS Topic](https://docs.aws.amazon.com/sns/latest/dg/sns-create-topic.html)

---

## When to Use Which

```
Fast inference (<60s)   →  Synchronous
Slow inference (>60s)   →  Asynchronous
Infrequent traffic      →  Asynchronous (scale to zero saves cost)
```


In [ ]:
sns_topic = "SNS_TOPIC_ARN"

async_config = {
    'ClientConfig': {
        'MaxConcurrentInvocationsPerInstance': 5
    },
    'OutputConfig': {
        'S3OutputPath': f"s3://{bucket}/async/out",
        'NotificationConfig': {
            'SuccessTopic': sns_topic,
            'ErrorTopic': sns_topic,
            'IncludeInferenceResponseIn': ['SUCCESS_NOTIFICATION_TOPIC']
        },
        'S3FailurePath': f"s3://{bucket}/async/err"
    }
}

config_response = sm_client.create_endpoint_config(
    EndpointConfigName=endpoint_config_name,
    ProductionVariants=[
        {
            "VariantName": variant_name,
            "ModelName": model_name,
            "InstanceType": instance["type"],
            "InitialInstanceCount": 1,
            "ContainerStartupHealthCheckTimeoutInSeconds": timeout,
        },
    ],
    AsyncInferenceConfig=async_config,
)

endpoint_response = sm_client.create_endpoint(
    EndpointName=endpoint_name, EndpointConfigName=endpoint_config_name
)

_ = sess.wait_for_endpoint(endpoint_name)

# WAN 2.2 Demo Payload — Parameter Rationale

## Hardware Constraints: `ml.g6e.2xlarge`

The `ml.g6e.2xlarge` instance is equipped with a single **NVIDIA L40S GPU**:

| Spec | Value |
|---|---|
| GPU | NVIDIA L40S |
| VRAM | 44.52 GB |
| Architecture | Ada Lovelace (sm_89) |
| vCPUs | 8 |
| System RAM | 32 GB |

While 44 GB sounds generous, the **WAN 2.2 TI2V-5B model alone consumes ~28–32 GB** after loading weights in `bfloat16`. This leaves only **12–16 GB of free headroom** for inference computation — which is where every constraint below originates.

---

## Why Each Parameter Was Chosen

### `num_frames: 49`

VRAM consumption scales with:
```
memory ∝ height × width × num_frames
```

This affects the transformer's **attention matrices** at every denoising step, not just the final output size. Our confirmed baseline was:

| Frames | Resolution | Outcome |
|---|---|---|
| 25 | 480×480 | ✅ ~22 GB, safe |
| 49 | 480×480 | ✅ ~38 GB, tight but works |
| 65 | 480×480 | ❌ ~46 GB, OOM |
| 81 | 480×480 | ❌ ~55 GB, definite OOM |

49 frames at 16fps produces a **~3 second clip** — long enough to show convincing motion, short enough to stay within the memory ceiling.

---

### `height: 480, width: 480`


We chose **480×480** for three reasons:

1. **It is a WAN 2.2 native resolution** — the model's spatial encoder was trained on this grid
2. **It fits within VRAM** — 960×960 (the ideal square resolution) caused an OOM at 49 frames, consuming an estimated ~52 GB
3. **It avoids aspect ratio distortion** — combined with a center-crop of the input image, the face is preserved without stretching

The tradeoff is reduced sharpness compared to 720×480 or 832×480, but those resolutions would require dropping to ~25 frames to stay within budget.

---

### `num_inference_steps: 50`

More denoising steps = higher quality, but also higher inference time. The relationship is roughly linear in time but has **diminishing quality returns past 50 steps**.

| Steps | Quality | Inference time |
|---|---|---|
| 20 | Acceptable | ~4 min |
| 30 | Good | ~6 min |
| 50 | Best practical | ~10 min |
| 80 | Marginal gain | ~16 min |

50 is the sweet spot for a demo — maximum perceivable quality without an unreasonable wait time.

---

### `guidance_scale: 7.5–8.5`

Guidance scale controls how strictly the model follows the text prompt vs. exercising creative freedom. For a demo we want **high prompt adherence** so the output matches expectations:

- Below **6.0** — model ignores parts of the prompt, output feels random
- **7.5–8.5** — strong prompt following, vivid and dramatic results ✅
- Above **10.0** — over-saturation, color artifacts, unnatural motion


---

### `fps: 16`

WAN 2.2 was trained at **16fps natively**. Using any other value means the model's learned temporal motion priors don't align with the output framerate, causing subtle stuttering or unnatural motion. Always use 16fps unless you explicitly resample in post-processing.

---

### `seed: fixed (42 / 7 / 123)`

A fixed seed is critical for a demo environment because:

- It makes results **reproducible** — you can re-run and get the same video
- It allows **A/B comparison** when tuning parameters
- It prevents accidentally shipping a bad generation to a stakeholder

In production you would randomize the seed or expose it as a parameter. For demos, pin it.

---

## Summary: The g6e.2xlarge Constraint Triangle

Every parameter decision above is a negotiation between three competing forces:
```
        Quality
           ▲
           │
           │
  VRAM ◄───┼───► Duration
  Budget        (num_frames)
```

The `ml.g6e.2xlarge` with its **~12–16 GB effective headroom** forces a compromise. To increase any one dimension you must reduce another:

- Want **longer video**? → Drop resolution from 480×480 to 320×320
- Want **higher resolution**? → Drop from 49 frames to 25
- Want **both**? → Upgrade to `ml.p4de.24xlarge` (A100 80GB), which has ~4× the effective headroom and can run 81 frames at 720×480 comfortably

In [ ]:
payload = f"""
{{
    "bucket": "{bucket}",
    "file_name": "demo_astronaut.mp4",
    "image_s3_uri":"s3://SOURCE_S3_BUCKET/astro.jpg",
    "prompt": "An astronaut in a white NASA spacesuit very slowly rotating and floating weightlessly in deep space, Earth's blue curvature visible in the background, golden sunlight gleaming off the visor, stars drifting gently, subtle breathing motion visible in the suit, cinematic IMAX footage, photorealistic, dramatic volumetric lighting",
    "negative_prompt": "static, frozen, cartoon, anime, blurry, deformed suit, bad anatomy, watermark, low quality, jerky motion, text, logo, fast movement",
    "height": 480,
    "width": 480,
    "num_frames": 49,
    "guidance_scale": 8.0,
    "num_inference_steps": 50,
    "fps": 25,
    "seed": 42
}}




"""
file_name = "request1.txt"
with open(file_name, 'w') as f:
    f.write(payload)

key = f"async/in/{file_name}"

s3.upload_file(file_name, bucket, key)

res = smr_client.invoke_endpoint_async(
    EndpointName = endpoint_name,
    ContentType = "application/json",
    InputLocation = f"s3://{bucket}/{key}",
)
print(payload)
print(json.dumps(res, indent=2))

In [ ]:
sess.delete_endpoint(endpoint_name)
sess.delete_endpoint_config(endpoint_config_name)
sess.delete_model(model_name)